In [ ]:
import os
import pickle
import json
import itertools
import time
import glob
import random
import copy
import json

import pywt
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
import pandas as pd
from joblib import Parallel, delayed

import scipy.io as sio
from scipy.interpolate import interp1d
import scipy.signal as signal

import optuna
from optuna.visualization import plot_optimization_history, plot_intermediate_values, plot_param_importances
from optuna.visualization import plot_contour, plot_slice
optuna.logging.set_verbosity(optuna.logging.WARNING)

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import MinMaxScaler

In [ ]:
dataset = "data_noise_vehicle_temperature"

In [ ]:
add_string = ''
if 'noise' in dataset:
    add_string += 'n'
if 'temperature' in dataset:
    add_string += 't'
if 'vehicle' in dataset:
    add_string += 'v'
if 'speed' in dataset:
    add_string += 's'
if add_string == '':
    add_string += 'all'

db_additive = "sqlite:///ttbi_additive_ablation_" + add_string + ".db"
experiment_name_additive = "results_additive_"  + add_string # The main folder for this run

db_DOFs_waterfall = "sqlite:///ttbi_DOFs_waterfall_ablation_" + add_string + ".db"
experiment_name_waterfall = "results_waterfall_"  + add_string # The main folder for this run

cache_dir = "data_cache_" + add_string

# Load dataset

In [ ]:
import scipy.io as sio
import numpy as np

def load_ttbi_dataset_v3(filepath, requested_dofs, n_passages=200):
    """
    requested_dofs: A list of integers from 0 to 7 mapping to the specific physical DOFs.
      0: Car Body Vert    (AcelPrimVag row 0)
      1: Front Bogie Vert (AcelPrimVag row 1)
      2: Rear Bogie Vert  (AcelPrimVag row 2)
      3: Wheel 1 Vert     (AcelRodaPrimVag row 0)
      4: Wheel 2 Vert     (AcelRodaPrimVag row 1)
      5: Car Body Pitch   (PitchPrimVag row 0)
      6: Front Bogie Pitch(PitchPrimVag row 1)
      7: Rear Bogie Pitch (PitchPrimVag row 2)
    """    
    dataset_path = os.path.join('data', filepath)
    if not os.path.exists(dataset_path):
        raise FileNotFoundError(f"Dataset folder not found: {dataset_path}")

    X_list = []
    y_list = []
    
    for damage_label in range(0, 61):
        # Format the file name, e.g., 0001.mat
        filename = f"{damage_label+1:04d}.mat"
        filepath = os.path.join(dataset_path, filename)
        
        try:
            mat = sio.loadmat(filepath)
            
            # Navigate the deeply nested MATLAB struct
            # mat['data'] is usually a 1x1 object array, containing the struct fields
            data_struct = mat['data'][0, 0]
            
            # Check how many passages actually exist in this file
            available_passages = data_struct['AcelPrimVag'].shape[1]
            passages_to_load = min(n_passages, available_passages)
            
            for p in range(passages_to_load):
                # Extract the full matrices for this passage
                acel_vag = data_struct['AcelPrimVag'][0, p]      # Shape: (3, seq_len)
                acel_roda = data_struct['AcelRodaPrimVag'][0, p] # Shape: (4, seq_len)
                pitch_vag = data_struct['PitchPrimVag'][0, p]    # Shape: (3, seq_len)
                
                passage_channels = []
                
                for dof in requested_dofs:
                    if dof == 0: passage_channels.append(acel_vag[0, :])
                    elif dof == 1: passage_channels.append(acel_vag[1, :])
                    elif dof == 2: passage_channels.append(acel_vag[2, :])
                    elif dof == 3: passage_channels.append(acel_roda[0, :])
                    elif dof == 4: passage_channels.append(acel_roda[1, :])
                    elif dof == 5: passage_channels.append(pitch_vag[0, :])
                    elif dof == 6: passage_channels.append(pitch_vag[1, :])
                    elif dof == 7: passage_channels.append(pitch_vag[2, :])
                        
                # Stack into (Channels, Sequence_Length)
                X_list.append(np.vstack(passage_channels))
                y_list.append(damage_label)
                
        except FileNotFoundError:
            print(f"  [!] Missing file: {filename}")
        except KeyError:
            print(f"  [!] Field not found in {filename}")
        except Exception as e:
            print(f"  [!] Error processing {filename}: {e}")

    # Convert to PyTorch-friendly NumPy arrays
    X = np.array(X_list, dtype=np.float32)
    y = np.array(y_list, dtype=np.int64) # int64 is standard for PyTorch classification labels
        
    return X, y

# Preprocessors

In [ ]:
class TTBIPreprocessor:
    def __init__(self, method='raw', n_segments=512, cwt_scales=64):
        """
        Initializes the preprocessor for the TTBI ablation study.
        
        Args:
            method (str): 'raw', 'paa', 'fft', or 'cwt'.
            n_segments (int): The target length for PAA downsampling (or FFT bin count).
            cwt_scales (int): Number of frequency scales for the Wavelet Transform.
        """
        self.method = method.lower()
        self.n_segments = n_segments
        self.cwt_scales = cwt_scales
        self.scaler = MinMaxScaler(feature_range=(0, 1))
        self.is_fit = False

    def _apply_paa(self, X):
        """Downsamples the sequence length using Piecewise Aggregate Approximation (Interpolation)."""
        samples, channels, length = X.shape
        if length == self.n_segments:
            return X
            
        x_old = np.linspace(0, 1, length)
        x_new = np.linspace(0, 1, self.n_segments)
        
        # Fully vectorized interpolation along the last axis (axis=2)
        f_interp = interp1d(x_old, X, axis=2, kind='linear')
        X_paa = f_interp(x_new)
        return X_paa.astype(np.float32)

    def _apply_fft(self, X):
        """Computes the frequency magnitude spectrum using Fast Fourier Transform."""
        # rfft automatically computes only the positive frequencies for real inputs
        fft_coeffs = np.fft.rfft(X, axis=2)
        fft_mag = np.abs(fft_coeffs)
        
        # If the resulting bins don't match our target, interpolate them
        return self._apply_paa(fft_mag)

    def _apply_cwt(self, X):
        """
        Generates 2D Scalograms using the Continuous Wavelet Transform (Morlet wavelet).
        Note: Applies PAA first to prevent RAM Out-Of-Memory crashes.
        """
        X_downsampled = self._apply_paa(X)
        samples, channels, length = X_downsampled.shape
        scales = np.arange(1, self.cwt_scales + 1)
        
        print(f"  -> Computing CWT in parallel (Shape: {samples}x{channels}x{self.cwt_scales}x{length})...")
        start_time = time.perf_counter()

        # Define a helper function to process one sample (all its channels)
        def process_single_sample(i):
            sample_cwt = np.zeros((channels, self.cwt_scales, length), dtype=np.float32)
            for c in range(channels):
                coeffs, freqs = pywt.cwt(X_downsampled[i, c, :], scales, 'morl')
                sample_cwt[c, :, :] = np.abs(coeffs)
            return sample_cwt
            
        # Run across all CPU cores simultaneously (n_jobs=-1)
        results = Parallel(n_jobs=-1, batch_size='auto')(
            delayed(process_single_sample)(i) for i in range(samples)
        )
        
        end_time = time.perf_counter()
        elapsed_time = end_time - start_time

        print(f"  -> Computed CWT in parallel in {elapsed_time:.4f} seconds")
        
        # Stack the parallel results back into a single tensor
        X_cwt = np.stack(results)
        return X_cwt

    def transform(self, X, fit_scaler=False):
        """
        Applies the selected signal processing method and scales the data.
        
        Args:
            X (np.ndarray): Input data of shape (Samples, Channels, Sequence_Length)
            fit_scaler (bool): If True, fits the MinMaxScaler. If False, only transforms.
            
        Returns:
            X_scaled (np.ndarray): Processed and scaled data ready for PyTorch.
        """
        # print(f"Applying '{self.method.upper()}' preprocessing...")
        
        # 1. Signal Processing
        if self.method == 'raw':
            X_processed = X
        elif self.method == 'paa':
            X_processed = self._apply_paa(X)
        elif self.method == 'fft':
            X_processed = self._apply_fft(X)
        elif self.method == 'cwt' or self.method == 'paa_cwt':
            X_processed = self._apply_cwt(X)
        else:
            raise ValueError(f"Unknown preprocessing method: {self.method}")

        # 2. Scaling (MinMaxScaler expects 2D data)
        # We flatten the Channels/Sequences into a single feature vector per sample
        original_shape = X_processed.shape
        samples = original_shape[0]
        
        X_flat = X_processed.reshape(samples, -1)
        
        if fit_scaler:
            X_scaled_flat = self.scaler.fit_transform(X_flat)
            self.is_fit = True
        else:
            if not self.is_fit:
                raise RuntimeError("Scaler has not been fitted yet! Pass fit_scaler=True for training data.")
            X_scaled_flat = self.scaler.transform(X_flat)
            
        # Reshape back to the 3D (or 4D for CWT) tensor shape expected by PyTorch CNNs
        X_scaled = X_scaled_flat.reshape(original_shape)
        
        return X_scaled

    def save_scaler(self, filepath):
        """Saves the fitted scaler for the Digital Twin online phase."""
        with open(filepath, 'wb') as f:
            pickle.dump(self.scaler, f)
        print(f"Scaler saved to {filepath}")

# Models

In [ ]:
# =====================================================================
# 1. Spatial Embedding Module
# =====================================================================
class Space2Vec(nn.Module):
    """
    Learnable spatial embedding layer (adapted from Time2Vec).
    Grounds the vibration signal to physical coordinates on the bridge.
    """
    def __init__(self, seq_len, out_features=8):
        super(Space2Vec, self).__init__()
        self.seq_len = seq_len
        self.out_features = out_features
        
        self.w_linear = nn.Parameter(torch.randn(1, 1))
        self.p_linear = nn.Parameter(torch.randn(1, 1))
        
        self.w_periodic = nn.Parameter(torch.randn(out_features - 1, 1))
        self.p_periodic = nn.Parameter(torch.randn(out_features - 1, 1))

    def forward(self, x_space):
        # x_space shape: (Batch, 1, Sequence_Length)
        linear = self.w_linear * x_space + self.p_linear
        periodic = torch.sin(self.w_periodic * x_space + self.p_periodic)
        return torch.cat([linear, periodic], dim=1) # Shape: (Batch, out_features, Seq_Len)

# =====================================================================
# 2. Multi-Rate Pooling Module (N-HiTS Style)
# =====================================================================
class MultiRatePooling1D(nn.Module):
    """
    Extracts features at multiple temporal resolutions by sub-sampling 
    the sequence at different pooling rates before the dense layers.
    """
    def __init__(self, pool_rates=[1, 2, 4]):
        super(MultiRatePooling1D, self).__init__()
        self.pool_rates = pool_rates

    def forward(self, x):
        # x shape expects: (Batch, Features, Sequence_Length)
        pooled_outputs = []
        for rate in self.pool_rates:
            if rate > 1:
                pooled = F.max_pool1d(x, kernel_size=rate, stride=rate)
            else:
                pooled = x
            
            # Flatten the pooled sequence 
            pooled_outputs.append(pooled.flatten(start_dim=1))
            
        # Concatenate all temporal resolutions together
        return torch.cat(pooled_outputs, dim=1) # Shape: (Batch, Flattened_Features)

# =====================================================================
# 3. The Ultimate Modular Network
# =====================================================================
class SpaceAwareModularNetwork(nn.Module):
    """
    Fully modular architecture supporting dynamic insertion of Space2Vec, 
    LSTM, and N-HiTS blocks for rigorous physical ablation studies.
    """
    def __init__(self, n_segments, n_classes, in_channels, params, 
                 use_space2vec=True, use_lstm=True, use_nhits=True, s2v_features=8):
        super(SpaceAwareModularNetwork, self).__init__()
        
        self.params = params
        self.use_space2vec = use_space2vec
        self.use_lstm = use_lstm
        self.use_nhits = use_nhits
        self.n_segments = n_segments
        
        # ---------------------------------------------------------
        # 1. Setup Space2Vec
        # ---------------------------------------------------------
        if self.use_space2vec:
            self.space2vec = Space2Vec(seq_len=n_segments, out_features=s2v_features)
            cnn_in_channels = in_channels + s2v_features
        else:
            cnn_in_channels = in_channels
            
        # ---------------------------------------------------------
        # 2. Build Dynamic CNN Layers
        # ---------------------------------------------------------
        self.cnn_layers = nn.ModuleList()
        current_seq_len = n_segments
        
        n_conv_layers = self.params.get('n_conv_layers', 2)
        for i in range(n_conv_layers):
            out_channels = self.params[f'n_filters_l{i}']
            kernel_size = self.params[f'kernel_size_l{i}']
            
            self.cnn_layers.append(nn.Conv1d(cnn_in_channels, out_channels, kernel_size=kernel_size, padding='same'))
            self.cnn_layers.append(nn.ReLU())
            
            if self.params.get(f'pooling_l{i}', False):
                self.cnn_layers.append(nn.MaxPool1d(kernel_size=2, stride=2))
                current_seq_len = current_seq_len // 2
                
            cnn_in_channels = out_channels
            
        # ---------------------------------------------------------
        # 3. Build Optional LSTM Layer
        # ---------------------------------------------------------
        if self.use_lstm:
            lstm_hidden = self.params.get('lstm_hidden_size', 64)
            lstm_layers = self.params.get('lstm_num_layers', 1)
            
            self.lstm = nn.LSTM(
                input_size=cnn_in_channels, 
                hidden_size=lstm_hidden, 
                num_layers=lstm_layers, 
                batch_first=True,
                dropout=self.params.get('lstm_dropout', 0.2) if lstm_layers > 1 else 0.0
            )
            current_features = lstm_hidden
        else:
            current_features = cnn_in_channels

        # ---------------------------------------------------------
        # 4. Build Optional N-HiTS Multi-Rate Pooling
        # ---------------------------------------------------------
        if self.use_nhits:
            # Dynamically grab the pool rates from Optuna (default to [1,2,4])
            pool_rates = self.params.get('nhits_pool_rates', (1, 2, 4))
            self.multi_rate_pool = MultiRatePooling1D(pool_rates=pool_rates)
            
            # Calculate the math for the Flattened dimension dynamically
            flattened_size = 0
            for rate in pool_rates:
                # Add the length of each sub-sampled sequence
                flattened_size += current_features * (current_seq_len // rate)
        else:
            # Fallback to standard Global Average Pooling
            flattened_size = current_features

        # ---------------------------------------------------------
        # 5. Build Dynamic Dense (Classification) Layers
        # ---------------------------------------------------------
        self.dense_layers = nn.ModuleList()
        n_dense_layers = self.params.get('n_dense_layers', 1)
        in_features = flattened_size 
        
        for i in range(n_dense_layers):
            out_features = self.params[f'n_dense_units_l{i}']
            self.dense_layers.append(nn.Linear(in_features, out_features))
            self.dense_layers.append(nn.ReLU())
            self.dense_layers.append(nn.Dropout(self.params.get(f'dropout_l{i}', 0.2)))
            in_features = out_features
            
        self.final_layer = nn.Linear(in_features, n_classes)

    # ---------------------------------------------------------
    # 6. The Forward Pass (Data Routing)
    # ---------------------------------------------------------
    def forward(self, x):
        batch_size = x.size(0)
        
        # 1. Space2Vec Pass
        if self.use_space2vec:
            space_vector = torch.linspace(0, 1, steps=self.n_segments, device=x.device)
            space_vector = space_vector.view(1, 1, -1).expand(batch_size, 1, -1)
            s2v_embeddings = self.space2vec(space_vector)
            x = torch.cat([x, s2v_embeddings], dim=1)
            
        # 2. CNN Pass -> Output shape: (Batch, Channels, SeqLen)
        for layer in self.cnn_layers:
            x = layer(x)
            
        # 3. LSTM Pass
        if self.use_lstm:
            # LSTM expects: (Batch, SeqLen, Features)
            x = x.permute(0, 2, 1) 
            x, _ = self.lstm(x)
            # Pooling expects: (Batch, Features, SeqLen)
            x = x.permute(0, 2, 1) 
            
        # 4. Pooling / Sub-sampling Pass
        if self.use_nhits:
            # N-HiTS outputs a flat 2D tensor: (Batch, Flattened_Features)
            x = self.multi_rate_pool(x)
        else:
            # Global Average Pooling outputs a flat 2D tensor: (Batch, Features)
            x = torch.mean(x, dim=2) 
            
        # 5. Dense Pass
        for layer in self.dense_layers:
            x = layer(x)
            
        return self.final_layer(x)


# =====================================================================
# 4. The 2D CNN for CWT
# =====================================================================
class Simple2DCNN(nn.Module):
    """
    Dynamic 2D CNN designed specifically for Continuous Wavelet Transform (CWT) scalograms.
    Input shape expects: (Batch_Size, Channels, Scales_Height, Sequence_Width)
    """
    def __init__(self, in_channels, n_classes, params, image_height=64, image_width=512):
        super(Simple2DCNN, self).__init__()
        self.params = params
        self.layers = nn.ModuleList()
        
        # Track the spatial dimensions to calculate the flatten size mathematically
        current_h = image_height
        current_w = image_width
        current_channels = in_channels
        
        n_conv_layers = self.params['n_conv_layers']
        
        # 1. Dynamic 2D Convolutional Layers
        for i in range(n_conv_layers):
            out_channels = self.params[f'n_filters_l{i}']
            # Optuna suggests a single integer (e.g., 3), which PyTorch interprets as a (3, 3) square kernel
            k_size = self.params[f'kernel_size_l{i}']
            
            # padding='same' ensures the convolution doesn't shrink the image dimensions
            self.layers.append(nn.Conv2d(current_channels, out_channels, kernel_size=k_size, padding='same'))
            self.layers.append(nn.ReLU())
            
            # 2x2 Max Pooling cuts the height and width exactly in half
            if self.params.get(f'pooling_l{i}', False):
                self.layers.append(nn.MaxPool2d(kernel_size=2, stride=2))
                current_h = current_h // 2
                current_w = current_w // 2
                
            current_channels = out_channels
            
        # 2. Flatten for the Dense Layers
        self.layers.append(nn.Flatten())
        flattened_size = current_channels * current_h * current_w
        
        # 3. Dynamic Dense (Linear) Layers
        n_dense_layers = self.params['n_dense_layers']
        in_features = flattened_size
        
        for i in range(n_dense_layers):
            out_features = self.params[f'n_dense_units_l{i}']
            self.layers.append(nn.Linear(in_features, out_features))
            self.layers.append(nn.ReLU())
            self.layers.append(nn.Dropout(self.params.get(f'dropout_l{i}', 0.2)))
            in_features = out_features
            
        # 4. Final Classification Output
        self.layers.append(nn.Linear(in_features, n_classes))
        
    def forward(self, x):
        """
        Args:
            x: Input scalograms of shape (Batch, Channels, Height, Width)
        """
        for layer in self.layers:
            x = layer(x)
        return x

# Training setup

In [ ]:
class MemmapDataset(torch.utils.data.Dataset):
    def __init__(self, X_memmap, y_memmap, indices):
        self.X = X_memmap
        self.y = y_memmap
        self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        # Get the true index from the shuffled list
        real_idx = self.indices[idx]
        
        # Load ONLY this specific sample into RAM and convert to PyTorch tensor
        # We use .copy() to ensure it doesn't hold a lock on the memmap array
        x_tensor = torch.tensor(self.X[real_idx].copy()).float()
        y_tensor = torch.tensor(self.y[real_idx].copy()).long()
        
        return x_tensor, y_tensor

In [ ]:
def get_or_create_cache(config, dataset_name, cache_dir):
    """
    Checks if the processed data exists. If yes, loads it instantly.
    If no, loads raw data, processes it, saves the cache, and returns it.
    """
    dof_str = "_".join(map(str, config['dofs']))
    cache_filename = os.path.join(cache_dir, f"cache_{dataset_name}_{config['method']}_dofs_{dof_str}.npy")
    labels_filename = os.path.join(cache_dir, f"cache_{dataset_name}_labels.npy")
    
    if os.path.exists(cache_filename) and os.path.exists(labels_filename):
        # FAST PATH
        X_processed = np.load(cache_filename)
        y = np.load(labels_filename)
    else:
        # SLOW PATH (Cache Miss)
        print(f"  [CACHE MISS] Processing and saving data for {config['method']} (DOFs: {dof_str})...")
        X_raw, y = load_ttbi_dataset_v3(
            filepath=dataset_name, 
            requested_dofs=config['dofs'], 
            n_passages=200
        )
        preprocessor = TTBIPreprocessor(method=config['method'], n_segments=512)
        X_processed = preprocessor.transform(X_raw, fit_scaler=True)
        
        # Save to disk
        np.save(cache_filename, X_processed)
        np.save(labels_filename, y)
        print(f"  [CACHE SAVED] Data successfully cached in {cache_dir}.")
        
        # Reload in memory-mapped mode to keep RAM usage identical
        X_processed = np.load(cache_filename, mmap_mode='r')
        y = np.load(labels_filename, mmap_mode='r')
        
    return X_processed, y

In [ ]:
# --- Hardware Setup ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =====================================================================
# 1. The Core Training & Evaluation Function
# =====================================================================
def train_and_evaluate(trial, config, dataset_name, n_epochs, cache_dir, output_dir):
    """
    Loads data, builds a dynamically sized model, trains it, and returns validation accuracy.
    Includes Optuna pruning to kill unpromising trials early.
    """
    # 1. Load Data (Will instantly load or create it)
    X_processed, y = get_or_create_cache(config, dataset_name, cache_dir)
    
    # 2. Train/Test Split
    # X_train, X_val, y_train, y_val = train_test_split(X_processed, y, test_size=0.2, random_state=42)
    
    # train_loader = DataLoader(TensorDataset(torch.tensor(X_train).float(), torch.tensor(y_train)), 
    #                           batch_size=32, shuffle=True)
    # val_loader = DataLoader(TensorDataset(torch.tensor(X_val).float(), torch.tensor(y_val)), 
    #                         batch_size=32, shuffle=False)
    
    all_indices = np.arange(len(y))
    
    train_idx, val_idx = train_test_split(all_indices, test_size=0.2, random_state=42)
    
    train_loader = DataLoader(
        MemmapDataset(X_processed, y, train_idx), 
        batch_size=32, 
        shuffle=True,
        num_workers=0 # Keep at 0 to avoid multiprocessing lock issues with memory maps
    )
    
    val_loader = DataLoader(
        MemmapDataset(X_processed, y, val_idx), 
        batch_size=32, 
        shuffle=False
    )

    # 3. Suggest Hyperparameters
    params = {
        'lr': trial.suggest_float('lr', 1e-4, 1e-2, log=True),
        'weight_decay': trial.suggest_float('weight_decay', 1e-5, 1e-3, log=True),
        'n_conv_layers': trial.suggest_int('n_conv_layers', 2, 4),
        'n_dense_layers': trial.suggest_int('n_dense_layers', 1, 3),
    }
    
    # Suggest CNN block parameters
    for i in range(params['n_conv_layers']):
        params[f'n_filters_l{i}'] = trial.suggest_int(f'n_filters_l{i}', 16, 128, step=16)
        params[f'kernel_size_l{i}'] = trial.suggest_categorical(f'kernel_size_l{i}', [2, 3, 5, 7])
        params[f'pooling_l{i}'] = trial.suggest_categorical(f'pooling_l{i}', [True, False])
    
    # Suggest dense block parameters    
    for i in range(params['n_dense_layers']):
        params[f'n_dense_units_l{i}'] = trial.suggest_int(f'n_dense_units_l{i}', 32, 256, step=16)
        params[f'dropout_l{i}'] = trial.suggest_float(f'dropout_l{i}', 0.1, 0.5)
    
    # ---------------------------------------------------------
    # NEW MODULAR CHECK: Only suggest LSTM params if the ablation 
    # configuration explicitly asks for the LSTM block.
    # ---------------------------------------------------------
    if config.get('use_lstm', False):
        params['lstm_hidden_size'] = trial.suggest_int('lstm_hidden_size', 32, 128, step=32)
        params['lstm_num_layers'] = trial.suggest_int('lstm_num_layers', 1, 2)
        if params['lstm_num_layers'] > 1:
            params['lstm_dropout'] = trial.suggest_float('lstm_dropout', 0.1, 0.4)
            
    # ---------------------------------------------------------
    # NEW MODULAR CHECK: N-HiTS Pooling Rates
    # ---------------------------------------------------------
    if config.get('use_nhits', False):
        # Define a dictionary mapping safe strings to the physical tuples
        pool_rate_options = {
            "1_2_4": (1, 2, 4),       # Standard gentle slope
            "1_4_8": (1, 4, 8),       # Aggressive low-frequency isolation
            "1_3_6": (1, 3, 6),       # Odd-numbered frequency intervals
            "1_2_4_8": (1, 2, 4, 8)   # Deep hierarchy (captures 4 distinct frequency bands)
        }
        
        # 1. Optuna safely suggests and stores the String key
        selected_key = trial.suggest_categorical(
            'nhits_pool_rates_key', 
            list(pool_rate_options.keys())
        )
        
        # 2. We extract the actual tuple and save it to params for the PyTorch network
        params['nhits_pool_rates'] = pool_rate_options[selected_key]

    # 4. Model Routing
    in_channels = X_processed.shape[1]
    
    if config['method'] == 'PAA_CWT':
        # Keep your Simple2DCNN routing here for the images
        scales_height = X_processed.shape[2]
        seq_width = X_processed.shape[3]
        model = Simple2DCNN(
            in_channels=in_channels,
            n_classes=61,
            params=params,
            image_height=scales_height, # <-- Pass the dynamic height
            image_width=seq_width       # <-- Pass the dynamic width
        ).to(DEVICE)
    else:
        # All 1D time-series methods (RAW, PAA) go through the Modular Network!
        seq_length = X_processed.shape[2]
        model = SpaceAwareModularNetwork(
            n_segments=seq_length,
            n_classes=61,
            in_channels=in_channels,
            params=params,
            use_space2vec=config.get('use_space2vec', True),
            use_lstm=config.get('use_lstm', False),
            use_nhits=config.get('use_nhits', False)
        ).to(DEVICE)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=params['lr'], weight_decay=params['weight_decay'])

    # 5. Training Loop with Pruning
    # Initialize the best Error to infinity (since we want to minimize it)
    best_val_error = float('inf')
    patience = 5  # How many epochs to wait before stopping
    patience_counter = 0
    
    for epoch in range(n_epochs):
        model.train()
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(DEVICE), batch_y.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
        # Validation
        model.eval()
        total_absolute_error = 0
        total = 0
        
        with torch.no_grad():
            for batch_X, batch_y in val_loader:
                batch_X, batch_y = batch_X.to(DEVICE), batch_y.to(DEVICE)
                outputs = model(batch_X)
                _, predicted = torch.max(outputs.data, 1)
                
                total += batch_y.size(0)
                # Calculate the class distance instead of strict accuracy
                # total_absolute_error += torch.abs(predicted - batch_y).sum().item()
                total_absolute_error += torch.pow(predicted - batch_y, 2).sum().item()
                
        val_error = total_absolute_error / total
                
        # 1. Optuna Pruning (Kills trials that are worse than the median Error)
        trial.report(val_error, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()
        
        # 2. Standard Early Stopping (Kills trials that have plateaued)
        # We now check if the Error has gone DOWN
        if val_error < best_val_error:
            best_val_error = val_error
            patience_counter = 0  # Reset counter
            
            # Save the model weights for THIS specific trial
            model_save_path = os.path.join(output_dir, f"weights_{config['name']}_trial_{trial.number}.pth")
            torch.save(model.state_dict(), model_save_path)
            
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break # Exit the epoch loop early!

    return best_val_error

# =====================================================================
# 2. Objective Wrapper for Optuna
# =====================================================================
class Objective:
    def __init__(self, config, dataset_name, n_epochs, cache_dir, output_dir):
        self.config = config
        self.dataset_name = dataset_name
        self.n_epochs = n_epochs
        self.cache_dir = cache_dir
        self.output_dir = output_dir

    def __call__(self, trial):
        return train_and_evaluate(trial, self.config, self.dataset_name, self.n_epochs,
                                  self.cache_dir, self.output_dir)

### Plot confusion matrices functions

In [ ]:
def plot_best_model_confusion_matrix(study_name, config, val_loader, db_path="sqlite:///BO_Additive_Noise_Temp_8DOFs.db"):
    """
    Rebuilds the best model from an Optuna study, evaluates it, and plots the Confusion Matrix.
    """
    # 1. Load the study and get the best hyperparameters
    study = optuna.load_study(study_name=study_name, storage=db_path)
    best_params = study.best_params
    
    print(f"Generating matrix for {study_name} with Accuracy: {study.best_value:.4f}")

    # 2. Rebuild the optimal architecture
    # (Assuming val_loader dataset shape is available to get in_channels and seq_length)
    sample_x, _ = next(iter(val_loader))
    in_channels = sample_x.shape[1]
    seq_length = sample_x.shape[2]
    
    if config.get('method') == 'PAA_CWT':
        scales_height = sample_x.shape[2]
        seq_width = sample_x.shape[3]
        model = Simple2DCNN(
            in_channels=in_channels,
            n_classes=61,
            params=best_params,
            image_height=scales_height,
            image_width=seq_width
        ).to(DEVICE)
    else:
        # Check if the study used N-HiTS strings and map them back to tuples
        if 'nhits_pool_rates_key' in best_params:
            pool_rate_options = {"1_2_4": (1, 2, 4), "1_4_8": (1, 4, 8), "1_3_6": (1, 3, 6), "1_2_4_8": (1, 2, 4, 8)}
            best_params['nhits_pool_rates'] = pool_rate_options[best_params['nhits_pool_rates_key']]

        model = SpaceAwareModularNetwork(
            n_segments=seq_length,
            n_classes=61,
            in_channels=in_channels,
            params=best_params,
            use_space2vec=config.get('use_space2vec', False),
            use_lstm=config.get('use_lstm', False),
            use_nhits=config.get('use_nhits', False)
        ).to(DEVICE)

    # Note: You would normally need to load the best saved model weights here (state_dict) 
    # if you saved them during training. If you didn't save weights, the network is initialized 
    # randomly, and you will need to quickly retrain this single best model for the n_epochs.
    
    # 3. Gather Predictions
    model.eval()
    all_preds = []
    all_trues = []
    
    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            batch_X, batch_y = batch_X.to(DEVICE), batch_y.to(DEVICE)
            outputs = model(batch_X)
            _, predicted = torch.max(outputs.data, 1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_trues.extend(batch_y.cpu().numpy())

    # 4. Plot the Confusion Matrix
    cm = confusion_matrix(all_trues, all_preds, labels=range(61))
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=False, cmap='Blues', cbar=True)
    
    plt.title(f'Confusion Matrix: {study_name.replace("_", " ")}', fontsize=16, fontweight='bold', pad=15)
    plt.xlabel('Predicted Scour Class', fontsize=12, fontweight='bold')
    plt.ylabel('True Scour Class', fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(f'CM_{study_name}.png', dpi=300)
    plt.show()

In [ ]:
def aggregate_confusion_matrix(cm):
    """
    Aggregates a 61x61 confusion matrix into a highly readable 4x4 matrix.
    Assumed Class Structure:
    - 0:     Healthy
    - 1-20:  Minor Scour
    - 21-40: Moderate Scour
    - 41-60: Severe Scour
    """
    # Define the slice indices for each category [start, end)
    bins = [(0, 1), (1, 21), (21, 41), (41, 61)]
    
    agg_cm = np.zeros((4, 4), dtype=int)
    
    for i, (row_start, row_end) in enumerate(bins):
        for j, (col_start, col_end) in enumerate(bins):
            # Sum the blocks of the original 61x61 matrix into a single cell
            agg_cm[i, j] = np.sum(cm[row_start:row_end, col_start:col_end])
            
    return agg_cm

def plot_aggregated_confusion_matrix(all_trues, all_preds, study_name, output_dir):
    """
    Takes the raw predictions, aggregates them, and plots a presentation-ready heatmap.
    """
    # 1. Generate the original 61x61 matrix
    cm_61 = confusion_matrix(all_trues, all_preds, labels=range(61))
    
    # 2. Aggregate down to 4x4
    cm_4 = aggregate_confusion_matrix(cm_61)
    
    # 3. Define the presentation-friendly labels
    # labels = ['Healthy\n(Class 0)', 'Minor\n(1-20)', 
    #           'Moderate\n(21-40)', 'Severe\n(41-60)']
    labels = range(61)
    
    # 4. Plot using Seaborn
    plt.figure(figsize=(16, 14))
    
    # Because it is only 4x4, we can use annot=True to print the exact numbers!
    # sns.heatmap(cm_4, annot=True, fmt='d', cmap='Blues', cbar=True,
    #             xticklabels=labels, yticklabels=labels, 
    #             annot_kws={"size": 16, "weight": "bold"})
    annot_labels = np.where(cm_61 > 0, cm_61.astype(str), "")

    ax = sns.heatmap(cm_61, annot=annot_labels, fmt='', cmap='Blues', cbar=True,
                     xticklabels=labels, yticklabels=labels, 
                     annot_kws={"size": 8, "weight": "bold"},
                     linewidths=0.2, linecolor='lightgray')
    
    # Draw a red bounding box around every cell where True == Predicted
    for i in range(61):
        ax.add_patch(patches.Rectangle((i, i), 1, 1, fill=False, edgecolor='#E74C3C', lw=2))
        
    plt.title(f'Risk Categorization: {study_name.replace("_", " ")}', 
              fontsize=16, fontweight='bold', pad=15)
    plt.xlabel('CNN Predicted Severity', fontsize=14, fontweight='bold')
    plt.ylabel('True Physical Severity', fontsize=14, fontweight='bold')
    
    plt.xticks(fontsize=8, rotation=90)
    plt.yticks(fontsize=8, rotation=0)
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f'Aggregated_CM_{study_name}.png'), dpi=300)
    plt.show()

In [ ]:
def plot_cached_confusion_matrix(study, config, dataset_name, cache_dir, output_dir):
    print(f"Generating matrix for {config['name']}...")
    
    # # 1. Load Data
    # X_processed, y = get_or_create_cache(config, dataset_name, cache_dir)
    
    # # Split data to get the exact same validation set
    # _, X_val, _, y_val = train_test_split(X_processed, y, test_size=0.2, random_state=42)
    # val_loader = DataLoader(TensorDataset(torch.tensor(X_val).float(), torch.tensor(y_val)), batch_size=32, shuffle=False)
    
    # 1. Load Data (Now returns memory-mapped disk arrays)
    X_processed, y = get_or_create_cache(config, dataset_name, cache_dir)
    
    # 2. SPLIT INDICES, NOT DATA
    # Create an array of indices [0, 1, 2 ... 12199]
    all_indices = np.arange(len(y))
    
    # Split the index numbers (Takes 0.0001 seconds and zero RAM)
    train_idx, val_idx = train_test_split(all_indices, test_size=0.2, random_state=42)
    
    # 3. Create DataLoaders using the custom MemmapDataset
    train_loader = DataLoader(
        MemmapDataset(X_processed, y, train_idx), 
        batch_size=32, 
        shuffle=True,
        num_workers=0 # Keep at 0 to avoid multiprocessing lock issues with memory maps
    )
    
    val_loader = DataLoader(
        MemmapDataset(X_processed, y, val_idx), 
        batch_size=32, 
        shuffle=False
    )

    # 2. Rebuild the blank optimal architecture
    best_params = study.best_params
    in_channels = X_processed.shape[1]
    
    if config['model_type'] == '2D_CNN':
        model = Simple2DCNN(in_channels=in_channels, n_classes=61, params=best_params, 
                            image_height=X_processed.shape[2], image_width=X_processed.shape[3]).to(DEVICE)
        # dummy_input = torch.randn(1, in_channels, X_processed.shape[2], X_processed.shape[3]).to(DEVICE)
    else:
        if 'nhits_pool_rates_key' in best_params:
            pool_map = {"1_2_4": (1, 2, 4), "1_4_8": (1, 4, 8), "1_3_6": (1, 3, 6), "1_2_4_8": (1, 2, 4, 8)}
            best_params['nhits_pool_rates'] = pool_map[best_params['nhits_pool_rates_key']]
            
        model = SpaceAwareModularNetwork(
            n_segments=X_processed.shape[2], n_classes=61, in_channels=in_channels, params=best_params,
            use_space2vec=config['use_space2vec'], use_lstm=config['use_lstm'], use_nhits=config['use_nhits']
        ).to(DEVICE)
        # dummy_input = torch.randn(1, in_channels, X_processed.shape[2]).to(DEVICE)

    # 3. Load the pre-trained weights from the BEST trial
    best_trial_number = study.best_trial.number
    model_save_path = os.path.join(output_dir, f"weights_{config['name']}_trial_{best_trial_number}.pth")
    model.load_state_dict(torch.load(model_save_path))
    
    # 4. Evaluate and Plot (No training required!)
    model.eval()
    all_preds, all_trues = [], []
    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            outputs = model(batch_X.to(DEVICE))
            _, predicted = torch.max(outputs.data, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_trues.extend(batch_y.numpy())
            
    plot_aggregated_confusion_matrix(all_trues, all_preds, config['name'], output_dir)
    
    # ==========================================
    # 5. ONNX EXPORT
    # ==========================================
    # onnx_file_path = os.path.join(output_dir, f"Architecture_{config['name']}.onnx")
    # torch.onnx.export(
    #     model, dummy_input, onnx_file_path,
    #     export_params=True, opset_version=14, do_constant_folding=True,
    #     input_names=['Sensor_Signals'], output_names=['Scour_Class'],
    #     dynamic_axes={'Sensor_Signals': {0: 'batch_size'}, 'Scour_Class': {0: 'batch_size'}}
    # )
    # print(f"--> Saved ONNX Architecture diagram.")
    
    # 6. Cleanup: Delete the .pth files for all the losing trials to save disk space
    search_pattern = os.path.join(output_dir, f"weights_{config['name']}_trial_*.pth")
    for file in glob.glob(search_pattern):
        if str(best_trial_number) not in file:
            os.remove(file)

# For Robustness Evaluation

In [ ]:
def set_random_seed(seed):
    """Locks down all sources of randomness for reproducibility."""
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def run_single_training(config, params, X_processed, y, seed, n_epochs):
    """Helper function to build, train, and evaluate a single model. Returns (Accuracy, Error)."""
    set_random_seed(seed)
    
    # 1. Shuffle and split data using the CURRENT seed
    # _, X_val, _, y_val = train_test_split(X_processed, y, test_size=0.2, random_state=42) 
    # X_train, _, y_train, _ = train_test_split(X_processed, y, test_size=0.8, random_state=seed) 
    
    # train_loader = DataLoader(TensorDataset(torch.tensor(X_train).float(), torch.tensor(y_train)), batch_size=32, shuffle=True)
    # val_loader = DataLoader(TensorDataset(torch.tensor(X_val).float(), torch.tensor(y_val)), batch_size=32, shuffle=False)
    
    # 1. SPLIT INDICES, NOT DATA (Memory Safe!)
    all_indices = np.arange(len(y))
    
    # Keep validation set perfectly static across all runs (Seed 42)
    _, val_idx = train_test_split(all_indices, test_size=0.2, random_state=42) 
    
    # Shuffle the training indices using the CURRENT run's seed
    train_idx, _ = train_test_split(all_indices, test_size=0.2, random_state=seed) 
    
    # 2. Create DataLoaders using the custom MemmapDataset
    train_loader = DataLoader(
        MemmapDataset(X_processed, y, train_idx), 
        batch_size=32, 
        shuffle=True,
        num_workers=0
    )
    val_loader = DataLoader(
        MemmapDataset(X_processed, y, val_idx), 
        batch_size=32, 
        shuffle=False
    )

    # 2. Build the model
    in_channels = X_processed.shape[1]
    if config['model_type'] == '2D_CNN':
        model = Simple2DCNN(in_channels=in_channels, n_classes=61, params=params, 
                            image_height=X_processed.shape[2], image_width=X_processed.shape[3]).to(DEVICE)
    else:
        local_params = copy.deepcopy(params)
        if 'nhits_pool_rates_key' in local_params:
            pool_map = {"1_2_4": (1, 2, 4), "1_4_8": (1, 4, 8), "1_3_6": (1, 3, 6), "1_2_4_8": (1, 2, 4, 8)}
            local_params['nhits_pool_rates'] = pool_map[local_params['nhits_pool_rates_key']]
            
        model = SpaceAwareModularNetwork(
            n_segments=X_processed.shape[2], n_classes=61, in_channels=in_channels, params=local_params,
            use_space2vec=config.get('use_space2vec', False), use_lstm=config.get('use_lstm', False), use_nhits=config.get('use_nhits', False)
        ).to(DEVICE)

    # 3. Train the model
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=params['lr'], weight_decay=params.get('weight_decay', 1e-4))
    
    model.train()
    for epoch in range(n_epochs):
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            outputs = model(batch_X.to(DEVICE))
            loss = criterion(outputs, batch_y.to(DEVICE))
            loss.backward()
            optimizer.step()

    # 4. Evaluate Accuracy and Mean Squared Error (Class Distance)
    model.eval()
    correct, total_absolute_error, total_squared_error, total = 0, 0, 0, 0
    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            batch_y_device = batch_y.to(DEVICE)
            outputs = model(batch_X.to(DEVICE))
            _, predicted = torch.max(outputs.data, 1)
            
            total += batch_y.size(0)
            correct += (predicted == batch_y_device).sum().item()
            # Calculate how many classes away the prediction was
            # The MAE Penalty (Linear)
            total_absolute_error += torch.abs(predicted - batch_y_device).sum().item()
            
            # The MSE Penalty (Exponential / Outlier Alarm)
            total_squared_error += torch.pow(predicted - batch_y_device, 2).sum().item()
            
    accuracy = correct / total
    mae = total_absolute_error / total
    mse = total_squared_error / total
    
    return accuracy, mae, mse

def evaluate_robustness(study, config, dataset_name, n_epochs=50, max_error_threshold=100.0, cache_dir='', output_dir=''):
    # For MAE max_error_threshold ~ 8.0
    # For MSE max_error_threshold ~ 100.0
    
    print(f"\n--> Evaluating Robustness for: {config['name']}...")
    
    if study.best_value > max_error_threshold:
        print(f"  [SKIP] Model's best error ({study.best_value:.2f}) is worse than the {max_error_threshold} threshold. Skipping robustness tests.")
        return

    best_params = study.best_params
    X_processed, y = get_or_create_cache(config, dataset_name, cache_dir)

    json_stoch_file = os.path.join(output_dir, f"robustness_stochastic.json")
    json_sens_file = os.path.join(output_dir, f"robustness_sensitivity.json")

    # ==========================================================
    # TEST 1: STOCHASTIC ROBUSTNESS (20-Seed)
    # ==========================================================
    if os.path.exists(json_stoch_file):
        print("  [CACHE HIT] Loading 20-Seed results from disk...")
        with open(json_stoch_file, 'r') as f:
            stoch_data = json.load(f)
            seed_accuracies, seed_maes, seed_mses = stoch_data['accuracies'], stoch_data['maes'], stoch_data['mses']
    else:
        print("  Running 20-Seed Stochastic Validation...")
        seed_accuracies, seed_maes, seed_mses = [], [], []
        for run in range(20):
            current_seed = 42 + run 
            val_acc, val_mae, val_mse = run_single_training(config, best_params, X_processed, y, seed=current_seed, n_epochs=n_epochs)
            seed_accuracies.append(val_acc); seed_maes.append(val_mae); seed_mses.append(val_mse)
            print(f"    Run {run+1}/20 (Seed {current_seed}): Acc = {val_acc:.4f} | MAE = {val_mae:.2f} | MSE = {val_mse:.2f}")
            
        with open(json_stoch_file, 'w') as f:
            json.dump({'accuracies': seed_accuracies, 'maes': seed_maes, 'mses': seed_mses}, f)

    # --- PLOT 1A: ACCURACY BOXPLOT ---
    plt.figure(figsize=(6, 8))
    sns.boxplot(y=seed_accuracies, color="#3498DB", width=0.4, fliersize=0)
    sns.swarmplot(y=seed_accuracies, color="black", alpha=0.6, size=6)
    plt.title(f"Stochastic Robustness: Accuracy\n{config['name']}", fontweight='bold', fontsize=14, pad=15)
    plt.ylabel("Validation Accuracy", fontweight='bold')
    plt.ylim(0, 1.0)
    plt.grid(axis='y', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"Robustness_Boxplot_Accuracy.png"), dpi=300)
    plt.close()

    # --- PLOT 1B: MAE BOXPLOT ---
    plt.figure(figsize=(6, 8))
    # Using a red color scheme since we are plotting an error (lower is better)
    sns.boxplot(y=seed_maes, color="#E74C3C", width=0.4, fliersize=0)
    sns.swarmplot(y=seed_maes, color="black", alpha=0.6, size=6)
    plt.title(f"Stochastic Robustness: Class Error (MAE)\n{config['name']}", fontweight='bold', fontsize=14, pad=15)
    plt.ylabel("Mean Absolute Error (Classes off)", fontweight='bold')
    plt.grid(axis='y', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"Robustness_Boxplot_MAE.png"), dpi=300)
    plt.close()
    
    # --- PLOT 1C: MSE BOXPLOT ---
    plt.figure(figsize=(6, 8))
    # Using a red color scheme since we are plotting an error (lower is better)
    sns.boxplot(y=seed_mses, color="#E74C3C", width=0.4, fliersize=0)
    sns.swarmplot(y=seed_mses, color="black", alpha=0.6, size=6)
    plt.title(f"Stochastic Robustness: Class Error (MSE)\n{config['name']}", fontweight='bold', fontsize=14, pad=15)
    plt.ylabel("Mean Squared Error", fontweight='bold')
    plt.grid(axis='y', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"Robustness_Boxplot_MSE.png"), dpi=300)
    plt.close()

    # ==========================================================
    # TEST 2: MULTI-PARAMETER SENSITIVITY (Separate Files)
    # ==========================================================
    if os.path.exists(json_sens_file):
        print("  [CACHE HIT] Loading Hyperparameter Sensitivity results from disk...")
        with open(json_sens_file, 'r') as f:
            perturbation_results = json.load(f)
    else:
        print("  Running Dynamic Hyperparameter Perturbations...")
        params_to_test = {'lr': best_params['lr'], 'weight_decay': best_params.get('weight_decay', 1e-4)}
        for p in ['dropout_l0', 'n_filters_l0', 'lstm_hidden_size']:
            if p in best_params: params_to_test[p] = best_params[p]

        perturbation_results = {}
        for param_name, base_val in params_to_test.items():
            perturbation_results[param_name] = {}
            for mult in [0.8, 0.9, 1.0, 1.1, 1.2]:
                perturbed_params = copy.deepcopy(best_params)
                new_val = max(1, int(base_val * mult)) if isinstance(base_val, int) else base_val * mult
                perturbed_params[param_name] = new_val
                
                val_acc, val_mae, val_mse = run_single_training(config, perturbed_params, X_processed, y, seed=42, n_epochs=n_epochs)
                
                # We now store a dictionary of ALL metrics for the sensitivity test
                perturbation_results[param_name][f"{mult*100:.0f}%"] = {'acc': val_acc, 'mae': val_mae, 'mse': val_mse}
                print(f"      {param_name} @ {mult}x: Acc = {val_acc:.4f} | MAE = {val_mae:.2f} | MSE = {val_mse:.2f}")

        with open(json_sens_file, 'w') as f:
            json.dump(perturbation_results, f)
            
    # --- PLOT 2: INDIVIDUAL SENSITIVITY FILES ---
    for param_name in perturbation_results.keys():
        plt.figure(figsize=(7, 5))
        
        x_labels = list(perturbation_results[param_name].keys())
        y_vals = list(perturbation_results[param_name].values())
        
        plt.plot(x_labels, y_vals, marker='o', linestyle='-', color='#E74C3C', linewidth=2)
        plt.axvline(x="100%", color='black', linestyle='--', label="Optimal BO Discovery")
        
        plt.title(f"Sensitivity: {param_name}\n{config['name']}", fontweight='bold', fontsize=14, pad=15)
        plt.xlabel("Parameter Multiplier", fontweight='bold')
        plt.ylabel("Validation Accuracy", fontweight='bold')
        plt.ylim(0, 1.0)
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        
        # Save each parameter to its own file!
        save_path = os.path.join(output_dir, f"Sensitivity_Perturbation_{param_name}.png")
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()
    
    print(f"--> Saved all robustness plots individually for {config['name']}.\n")
    
    # ==========================================================
    # 3. CALCULATE AND RETURN ROBUSTNESS METRICS
    # ==========================================================
    # 1. Stochastic Risk (Fernandes)
    mu_mse = np.mean(seed_mses)
    std_mse = np.std(seed_mses)
    ucb_mse = mu_mse + (2 * std_mse) # The 95% Confidence Upper Bound
    
    # 2. Parametric Risk (Todd)
    # Find the worst MSE across all perturbations
    worst_perturbed_mse = 0
    for param_data in perturbation_results.values():
        for metrics in param_data.values():
            if metrics['mse'] > worst_perturbed_mse:
                worst_perturbed_mse = metrics['mse']
                
    baseline_mse = np.mean(seed_mses) # The expected baseline
    max_degradation = worst_perturbed_mse - baseline_mse

    # Create the final scorecard
    robustness_scorecard = {
        'Optuna_Lucky_Score': study.best_value,
        'Stochastic_Mean_MSE': mu_mse,
        'Stochastic_Std_MSE': std_mse,
        'UCB_95_MSE': ucb_mse, 
        'Todd_Worst_MSE': worst_perturbed_mse,
        'Todd_Max_Degradation': max_degradation,
        # THE NEW UNIFIED METRIC
        'Global_Risk_Score': ucb_mse + max_degradation 
    }
    
    print(f"\n  [ROBUSTNESS SCORECARD]")
    print(f"  Optuna Baseline: {study.best_value:.2f}")
    print(f"  95% Confidence Error (UCB): {ucb_mse:.2f} MSE")
    print(f"  Worst Parametric Degradation: +{max_degradation:.2f} MSE")
    print(f"  --> GLOBAL RISK SCORE: {robustness_scorecard['Global_Risk_Score']:.2f}")
    
    return robustness_scorecard

In [ ]:
def generate_optuna_robustness_plots(study, config, output_dir='.'):
    print(f"--> Generating Individual Landscape Plots for {config['name']}...")
    
    df = study.trials_dataframe()
    df = df[df['state'] == 'COMPLETE']
    param_cols = [col for col in df.columns if col.startswith('params_')]
    
    if not param_cols:
        print("  [WARNING] No parameters found in DataFrame to plot.")
        return

    # Create a dedicated subfolder just for these slice plots!
    slice_dir = os.path.join(output_dir, "Slice_Plots")
    os.makedirs(slice_dir, exist_ok=True)

    # Plot and save each parameter individually
    for col in param_cols:
        param_name = col.replace('params_', '')
        
        plt.figure(figsize=(8, 6))
        ax = plt.gca()
        
        # Scatter the trials
        sns.scatterplot(data=df, x=col, y='value', ax=ax, color='#3498DB', alpha=0.6, s=80, edgecolor='black')
        
        # Highlight the winning configuration
        best_val = study.best_value
        best_param = study.best_params.get(param_name)
        if best_param is not None:
            ax.scatter([best_param], [best_val], color='#E74C3C', marker='*', s=400, edgecolor='black', zorder=5, label='Best Config')

        # Formatting
        plt.title(f"Sensitivity: {param_name}\n({config['name']})", fontweight='bold', fontsize=16, pad=15)
        plt.xlabel(f"{param_name} Value", fontweight='bold', fontsize=14)
        plt.ylabel("Validation Error (Lower is Better)", fontweight='bold', fontsize=14)
        
        if param_name in ['lr', 'weight_decay']:
            plt.xscale('log')
            
        plt.grid(True, alpha=0.3)
        if best_param is not None:
            plt.legend()

        plt.tight_layout()
        
        # Save to the new subfolder!
        save_path = os.path.join(slice_dir, f"Slice_{param_name}.png")
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()
        
    print(f"--> Saved {len(param_cols)} individual Slice Plots to {slice_dir}\n")

# Ablation

In [ ]:
# =====================================================================
# THE ADDITIVE ABLATION PATH (CONSTRUCTIVE BUILD-UP)
# =====================================================================

# We define a linear sequence, adding one physical/math component at a time
additive_path = [
    {
        "name": "1_Baseline_RAW_CNN",
        "method": "RAW",
        "dofs": list(range(8)),
        "use_space2vec": False,
        "use_lstm": False,
        "use_nhits": False,
        "model_type": "1D_MODULAR"
    },
    {
        "name": "2_Add_PAA_(Fernandes_2025)",
        "method": "PAA",
        "dofs": list(range(8)),
        "use_space2vec": False,
        "use_lstm": False,
        "use_nhits": False,
        "model_type": "1D_MODULAR"
    },
    {
        "name": "3_Add_Space2Vec",
        "method": "PAA",
        "dofs": list(range(8)),
        "use_space2vec": True,
        "use_lstm": False,
        "use_nhits": False,
        "model_type": "1D_MODULAR"
    },
    {
        "name": "4_Add_LSTM",
        "method": "PAA",
        "dofs": list(range(8)),
        "use_space2vec": True,
        "use_lstm": True,
        "use_nhits": False,
        "model_type": "1D_MODULAR"
    },
    {
        "name": "5_Add_NHITS",
        "method": "PAA",
        "dofs": list(range(8)),
        "use_space2vec": True,
        "use_lstm": True,
        "use_nhits": True,
        "model_type": "1D_MODULAR"
    },
    {
        "name": "6_Add_CWT_(Full_Model)",
        "method": "PAA_CWT",
        "dofs": list(range(8)),
        "use_space2vec": True,
        "use_lstm": True,
        "use_nhits": True,
        "model_type": "2D_CNN" 
    }
]

In [ ]:
# additive_path = [
#     # ==========================================
#     # GRUPO 1: DADOS RAW (Sinais Brutos)
#     # ==========================================
#     {
#         "name": "1_RAW_NHiTS_Puro",
#         "method": "RAW",
#         "dofs": list(range(8)),
#         "use_space2vec": False,
#         "use_lstm": False,
#         "use_nhits": True,
#         "model_type": "1D_MODULAR"
#     },
#     {
#         "name": "2_RAW_Space2Vec_NHiTS",
#         "method": "RAW",
#         "dofs": list(range(8)),
#         "use_space2vec": True,
#         "use_lstm": False,
#         "use_nhits": True,
#         "model_type": "1D_MODULAR"
#     },
#     {
#         "name": "3_RAW_LSTM_NHiTS",
#         "method": "RAW",
#         "dofs": list(range(8)),
#         "use_space2vec": False,
#         "use_lstm": True,
#         "use_nhits": True,
#         "model_type": "1D_MODULAR"
#     },
#     {
#         "name": "4_RAW_Completo", # Space2Vec + LSTM + NHiTS
#         "method": "RAW",
#         "dofs": list(range(8)),
#         "use_space2vec": True,
#         "use_lstm": True,
#         "use_nhits": True,
#         "model_type": "1D_MODULAR"
#     },

#     # ==========================================
#     # GRUPO 2: DADOS PAA (Suavizados)
#     # ==========================================
#     {
#         "name": "5_PAA_NHiTS_Puro",
#         "method": "PAA",
#         "dofs": list(range(8)),
#         "use_space2vec": False,
#         "use_lstm": False,
#         "use_nhits": True,
#         "model_type": "1D_MODULAR"
#     },
#     {
#         "name": "6_PAA_Space2Vec_NHiTS",
#         "method": "PAA",
#         "dofs": list(range(8)),
#         "use_space2vec": True,
#         "use_lstm": False,
#         "use_nhits": True,
#         "model_type": "1D_MODULAR"
#     },
#     {
#         "name": "7_PAA_LSTM_NHiTS",
#         "method": "PAA",
#         "dofs": list(range(8)),
#         "use_space2vec": False,
#         "use_lstm": True,
#         "use_nhits": True,
#         "model_type": "1D_MODULAR"
#     },
#     {
#         "name": "8_PAA_Completo", # Space2Vec + LSTM + NHiTS
#         "method": "PAA",
#         "dofs": list(range(8)),
#         "use_space2vec": True,
#         "use_lstm": True,
#         "use_nhits": True,
#         "model_type": "1D_MODULAR"
#     }
# ]

In [ ]:
# Create global cache directory
# cache_dir = "data_caches"
os.makedirs(cache_dir, exist_ok=True)

# Create summary folder for the final bar charts
summary_dir = os.path.join(experiment_name_additive, "Summary_Plots")
os.makedirs(summary_dir, exist_ok=True)

all_model_results = []
for step in additive_path:
    print(f"\n{'='*50}")
    print(f"Executing: {step['name']}")
    print(f"{'='*50}")
    
    # Create the specific folder for this model
    output_dir = os.path.join(experiment_name_additive, step['name'])
    os.makedirs(output_dir, exist_ok=True)
    
    # Adjust Optuna trials based on computational cost
    n_trials = 50
    epochs = 50

    study = optuna.create_study(
        study_name=step['name'],
        storage=db_additive,
        direction='minimize',
        load_if_exists=True
    )
    
    # 1. OPTIMIZATION
    # We only run trials if the study hasn't finished yet
    if len(study.trials) < n_trials:
        objective = Objective(config=step, dataset_name=dataset, n_epochs=epochs, 
                              cache_dir=cache_dir, output_dir=output_dir)
        study.optimize(objective, n_trials=(n_trials - len(study.trials)))
    
    # 2. EVALUATION & REPORTING (Includes ONNX Export)
    plot_cached_confusion_matrix(study=study, config=step, dataset_name=dataset, 
                                 cache_dir=cache_dir, output_dir=output_dir)
    
    # 3. ROBUSTNESS STRESS-TEST
    # We pass n_epochs=50 so the robustness test trains the networks fully
    robustness_metrics = evaluate_robustness(study=study, config=step, dataset_name=dataset, n_epochs=epochs, 
                        cache_dir=cache_dir, output_dir=output_dir)
    
    if robustness_metrics:
        # Save this to a master list to compare models at the end!
        all_model_results.append({
            'Model': step['name'],
            **robustness_metrics
        })
    
    generate_optuna_robustness_plots(study=study, config=step, output_dir=output_dir)
    
    print(f"Best Val Error for {step['name']}: {study.best_value:.4f} classes off")

In [ ]:
# Load the best errors in the exact order of the ablation path
names = []
errors = []

for step in additive_path: 
    study = optuna.load_study(study_name=step['name'], storage=db_additive)
    names.append(step['name'].replace("_", " "))
    errors.append(study.best_value)

# Create the plot
plt.figure(figsize=(14, 7))

# Plot a line connecting the points to show the "trend" of improvement/degradation
plt.plot(names, errors, color='black', linestyle='dashed', marker='o', alpha=0.5, zorder=3)

# Plot the bars
bars = plt.bar(names, errors, color='#3498DB', edgecolor='black', zorder=2)

# ---------------------------------------------------------
# DYNAMIC HIGHLIGHTING (Error metrics = Lower is Better)
# ---------------------------------------------------------
best_idx = np.argmin(errors)  # Finds the index of the LOWEST error
worst_idx = np.argmax(errors) # Finds the index of the HIGHEST error

bars[best_idx].set_color('#2ECC71')  # Green for the Best Model (Lowest Bar)
bars[worst_idx].set_color('#E74C3C') # Red for the Worst Model (Highest Bar)
bars[best_idx].set_edgecolor('black')
bars[worst_idx].set_edgecolor('black')

# Formatting
plt.title('Ablation Grid: Impact of Architectures on Prediction Error', fontsize=16, fontweight='bold', pad=20)
plt.ylabel('Validation Error (Lower is Better)', fontsize=14, fontweight='bold')
plt.xticks(rotation=30, ha='right', fontsize=11)

max_error = max(errors)
# plt.ylim(0, max_error * 1.15) # Set the top limit to 15% higher than the worst model so labels don't get cut off
plt.yscale('log')

plt.grid(axis='y', linestyle='--', alpha=0.7, zorder=0)

# Add data labels dynamically
for bar in bars:
    yval = bar.get_height()
    # Pushes the text slightly above the bar based on the dynamic maximum
    plt.text(bar.get_x() + bar.get_width()/2, yval + (max_error * 0.02), 
             f'{yval:.2f}', ha='center', va='bottom', 
             fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(summary_dir, 'ablation_summary_plot.png'), dpi=300)
plt.show()

# Leave-One-Out Ablation by removing the DOFs from the best model (check sensor importance)

In [ ]:
# =====================================================================
# SENSOR ABLATION: BACKWARD ELIMINATION (LEAVE-ONE-OUT)
# =====================================================================

# 1. AUTOPILOT: Find the champion model from the previous ablation
print("\n" + "="*50)
print("PHASE 2: SENSOR ABLATION (LEAVE-ONE-OUT)")
print("="*50)

# Find the dictionary in all_model_results that has the lowest score
# champion_result = min(all_model_results, key=lambda x: x['UCB_95_MSE'])
champion_result = min(all_model_results, key=lambda x: x['Global_Risk_Score'])
champion_name = champion_result['Model']

print(f"--> Champion Model Selected: {champion_name}")
print(f"    UCB 95% MSE Score: {champion_result['UCB_95_MSE']:.4f}")

# Look up the champion's original configuration dictionary from the ablation grid
champion_config = next(config for config in additive_path if config["name"] == champion_name)

# Define our universal baseline
base_dofs = [0, 1, 2, 3, 4, 5, 6, 7]

dof_names = [
    "CarBody_Vert",       # 0
    "FrontBogie_Vert",    # 1
    "RearBogie_Vert",     # 2
    "Wheel1_Vert",        # 3
    "Wheel2_Vert",        # 4
    "CarBody_Pitch",      # 5
    "FrontBogie_Pitch",   # 6
    "RearBogie_Pitch"     # 7
]

DOFs_waterfall_path = []

# 2. Build the new Baseline using the Champion's DNA
DOFs_waterfall_path.append({
    "name": f"Baseline_All_8_{champion_name}",
    "method": champion_config['method'],
    "use_lstm": champion_config['use_lstm'],
    "use_nhits": champion_config['use_nhits'],
    "use_space2vec": champion_config['use_space2vec'],
    "model_type": champion_config['model_type'],
    "dofs": base_dofs.copy(),
})

# 3. Backward Elimination (Drop exactly one sensor)
for i in range(len(base_dofs)):
    current_dofs = base_dofs.copy()
    dropped_dof = current_dofs.pop(i)
    
    DOFs_waterfall_path.append({
        "name": f"Drop_{dof_names[i]}",
        "method": champion_config['method'],
        "use_lstm": champion_config['use_lstm'],
        "use_nhits": champion_config['use_nhits'],
        "use_space2vec": champion_config['use_space2vec'],
        "model_type": champion_config['model_type'],
        "dofs": current_dofs
    })

print(f"\nGenerated {len(DOFs_waterfall_path)} configurations for Backward Elimination.")
for config in DOFs_waterfall_path:
    print(f" - {config['name']} (Using {len(config['dofs'])} DOFs)")

In [ ]:
# Create global cache directory
cache_dir = "data_caches"
os.makedirs(cache_dir, exist_ok=True)

# Create summary folder for the final bar charts
summary_dir = os.path.join(experiment_name_waterfall, "Summary_Plots")
os.makedirs(summary_dir, exist_ok=True)

for step in DOFs_waterfall_path:
    print(f"\n{'='*50}")
    print(f"Executing: {step['name']}")
    print(f"{'='*50}")
    
    # Adjust Optuna trials based on computational cost
    n_trials = 50
    epochs = 50
    
    # Create the specific folder for this model
    output_dir = os.path.join(experiment_name_waterfall, step['name'])
    os.makedirs(output_dir, exist_ok=True)

    study = optuna.create_study(
        study_name=step['name'],
        storage=db_DOFs_waterfall,
        direction='minimize',
        load_if_exists=True
    )
    
    # 1. OPTIMIZATION
    # We only run trials if the study hasn't finished yet
    if len(study.trials) < n_trials:
        objective = Objective(config=step, dataset_name=dataset, n_epochs=epochs, 
                              cache_dir=cache_dir, output_dir=output_dir)
        study.optimize(objective, n_trials=(n_trials - len(study.trials)))
    
    # 2. EVALUATION & REPORTING (Includes ONNX Export)
    plot_cached_confusion_matrix(study=study, config=step, dataset_name=dataset, 
                                 cache_dir=cache_dir, output_dir=output_dir)
    
    # 3. ROBUSTNESS STRESS-TEST
    # We pass n_epochs=50 so the robustness test trains the networks fully
    evaluate_robustness(study=study, config=step, dataset_name=dataset, n_epochs=epochs, 
                        cache_dir=cache_dir, output_dir=output_dir)
    
    generate_optuna_robustness_plots(study=study, config=step, output_dir=output_dir)
            
    print(f"Best Val Error for {step['name']}: {study.best_value:.4f} classes off")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
import optuna

# Load the best errors in the exact order of the ablation path
names = []
errors = []

# Make sure to use your new ablation_grid_path here!
for step in DOFs_waterfall_path: 
    study = optuna.load_study(study_name=step['name'], storage=db_additive)
    names.append(step['name'].replace("_", " "))
    errors.append(study.best_value)

# Create the plot
plt.figure(figsize=(14, 7))

# Plot a line connecting the points to show the "trend" of improvement/degradation
plt.plot(names, errors, color='black', linestyle='dashed', marker='o', alpha=0.5, zorder=3)

# Plot the bars
bars = plt.bar(names, errors, color='#3498DB', edgecolor='black', zorder=2)

# ---------------------------------------------------------
# DYNAMIC HIGHLIGHTING (Error metrics = Lower is Better)
# ---------------------------------------------------------
best_idx = np.argmin(errors)  # Finds the index of the LOWEST error
worst_idx = np.argmax(errors) # Finds the index of the HIGHEST error

bars[best_idx].set_color('#2ECC71')  # Green for the Best Model (Lowest Bar)
bars[worst_idx].set_color('#E74C3C') # Red for the Worst Model (Highest Bar)
bars[best_idx].set_edgecolor('black')
bars[worst_idx].set_edgecolor('black')

# Formatting
plt.title('Ablation Grid: Impact of Architectures on Prediction Error', fontsize=16, fontweight='bold', pad=20)
plt.ylabel('Validation Error (Lower is Better)', fontsize=14, fontweight='bold')
plt.xticks(rotation=30, ha='right', fontsize=11)

max_error = max(errors)
# plt.ylim(0, max_error * 1.15) # Set the top limit to 15% higher than the worst model so labels don't get cut off
plt.yscale('log')

plt.grid(axis='y', linestyle='--', alpha=0.7, zorder=0)

# Add data labels dynamically
for bar in bars:
    yval = bar.get_height()
    # Pushes the text slightly above the bar based on the dynamic maximum
    plt.text(bar.get_x() + bar.get_width()/2, yval + (max_error * 0.02), 
             f'{yval:.2f}', ha='center', va='bottom', 
             fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(summary_dir, 'ablation_summary_plot.png'), dpi=300)
plt.show()